# SupportBot - data exploration

Exploratory data analysis of the **Bitext customer-support dataset** (training data) and the
**CLINC150 out-of-scope queries** (used only to test out-of-scope detection).

Run `python src/data_download.py` first. Then *Run All*.

In [ ]:
import sys, re
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from common import tokenize
import plots  # shared plot style

DATA = ROOT / "data"
raw = pd.read_csv(DATA / "bitext_raw.csv")
print(raw.shape)
raw.head()

## 1. Columns and missing values

In [ ]:
print(raw.dtypes, "\n")
print("missing values:\n", raw.isna().sum())
print("\nexact duplicate messages:", raw["instruction"].duplicated().sum())

## 2. Intents and categories
Is the dataset balanced? (Balanced classes mean accuracy and macro-F1 will be close.)

In [ ]:
print(raw["category"].nunique(), "categories,", raw["intent"].nunique(), "intents")
counts = raw["intent"].value_counts().sort_values()
fig, ax = plt.subplots(figsize=(8, 8))
ax.barh(counts.index, counts.values, color=plots.SERIES[0], height=0.7)
ax.set(title="Messages per intent", xlabel="number of messages")
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()
print("smallest / largest intent:", counts.min(), "/", counts.max())

In [ ]:
raw.groupby("category")["intent"].unique().apply(lambda x: ", ".join(sorted(x))).to_frame("intents")

## 3. Message length
The from-scratch Transformer uses `max_len = 32` tokens - check this covers almost all messages.

In [ ]:
raw["n_tokens"] = raw["instruction"].apply(lambda t: len(tokenize(t)))
print(raw["n_tokens"].describe())
print("share of messages with <= 32 tokens:", (raw["n_tokens"] <= 32).mean().round(4))
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(raw["n_tokens"], bins=40, color=plots.SERIES[0])
ax.axvline(32, color=plots.INK, ls="--", lw=1.5)
ax.set(title="Tokens per customer message", xlabel="tokens", ylabel="messages")
plt.tight_layout(); plt.show()

## 4. Example messages per intent

In [ ]:
raw.groupby("intent").head(2)[["intent", "instruction"]].sort_values("intent").head(20)

## 5. Linguistic variation flags

Bitext tags each message with flags describing its style, e.g. **B** basic, **L** colloquial,
**Q** question, **P** polite, **Z** typos/errors, **M** morphological variation, **W** offensive.

In [ ]:
flag_counts = pd.Series([f for fl in raw["flags"].astype(str) for f in fl]).value_counts()
flag_counts.to_frame("messages with this flag")

In [ ]:
print("messages with typos (Z flag):")
raw[raw["flags"].astype(str).str.contains("Z")]["instruction"].head(8).tolist()

## 6. Responses and placeholders
Replies contain slots such as `{{Order Number}}`. The app shows them as `[Order Number]`.

In [ ]:
slots = raw["response"].str.findall(r"\{\{(.*?)\}\}").explode().value_counts()
print("different placeholders:", len(slots))
slots.head(15).to_frame("count")

In [ ]:
raw["response_words"] = raw["response"].str.split().str.len()
raw["response_words"].describe()

## 7. Vocabulary

In [ ]:
from collections import Counter
vocab = Counter(tok for t in raw["instruction"] for tok in tokenize(t))
print("vocabulary size:", len(vocab))
print("words that appear once:", sum(1 for c in vocab.values() if c == 1))
vocab.most_common(25)

## 8. Train / validation / test split (stratified, seed 42)

In [ ]:
splits = {n: pd.read_csv(DATA / f"{n}.csv") for n in ("train", "val", "test")}
pd.DataFrame({n: [len(d), d["intent"].nunique()] for n, d in splits.items()}, index=["rows", "intents"])

In [ ]:
# overlap check: the same message must not be in train and test
print("test messages also in train:", splits["test"]["text"].isin(splits["train"]["text"]).sum())

## 9. Out-of-scope queries (CLINC150)

These are *not* customer-support requests. The model never trains on them; they are used only to
measure whether SupportBot can say "this is not something I handle".

In [ ]:
oos_val, oos_test = pd.read_csv(DATA / "oos_val.csv"), pd.read_csv(DATA / "oos_test.csv")
print(len(oos_val), "validation /", len(oos_test), "test")
oos_test.sample(10, random_state=0)

## 10. Observations

* 27 intents in 11 categories, roughly balanced → accuracy and macro-F1 behave similarly.
* Messages are short; 32 tokens covers almost every message.
* Many paraphrases of the same request → in-distribution accuracy will be very high, so the
  more informative tests are **typos (noisy test set)** and **out-of-scope detection**.
* Replies contain placeholders that a real system fills from the order database.